# Application d'analyse financière assistée par IA

Cette application combine l'analyse financière traditionnelle avec l'IA pour :
- Analyser des données boursières (cours, métriques) de façon automatisée
- Générer des recommandations d'investissement personnalisées via l'API OpenAI
- Produire des rapports détaillés avec métriques, signaux et conseils
- Backtester des stratégies simples (ex: moyennes mobiles)

L'application utilise Python, l'API OpenAI et une interface Gradio pour permettre aux utilisateurs d'obtenir facilement des analyses financières pertinentes et contextualisées.


# Préparation et importations

In [1]:
!pip install yfinance openai pandas numpy matplotlib gradio

In [10]:

import matplotlib
matplotlib.use("Agg")  # backend non-GUI

import os, json
import datetime as dt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt  # requis (même si non affiché)
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI

# Fonctions locales
from fonctions import (
    load_prices, compute_metrics, rules_engine,
    render_report_md, build_ia_context, generate_advice_openai,
    detect_tickers, resolve_company_names
)

# -------------------------------
# Paramètres utilisateur
TICKERS = ["AAPL", "MSFT", "GOOGL"]
HORIZON = 12
PROFIL = "equilibre"
END_DATE = dt.date.today()
START_DATE = END_DATE - dt.timedelta(days=HORIZON*30)
DISCLAIMER = "Contenu éducatif. Pas un conseil en investissement."


In [11]:

# Chargement des prix et calcul des métriques
prices = load_prices(UNIVERSE, START_DATE, END_DATE)
metrics = compute_metrics(prices)


In [12]:

# 3) Règles
signals, weights = rules_engine(metrics, PROFIL)


In [13]:

# 4) Rapport déterministe
report = render_report_text(
    TICKERS, PROFIL, metrics, signals, weights,
    disclaimer=DISCLAIMER, horizon=HORIZON, date=END_DATE
)
print(report)




RAPPORT D'ANALYSE - 20/09/2025
Profil: equilibre
Horizon: 12 mois

=== AAPL ===
Signal: {'action': '50% DCA + 50% lump sum', 'stop': -0.12, 'tp': 0.2}
Poids suggéré: 25.0%

=== MSFT ===
Signal: {'action': '50% DCA + 50% lump sum', 'stop': -0.12, 'tp': 0.2}
Poids suggéré: 25.0%

=== GOOGL ===


Contenu éducatif. Pas un conseil en investissement.


In [17]:
# 5) Conseil IA — sans condition sur la clé

from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()  # si déjà fait plus haut, vous pouvez l’enlever
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Noms officiels (si pas déjà calculés)
names_by_ticker = resolve_company_names(list(metrics["ticker"]))

# Contexte IA (signature : profil, horizon, date, metrics, signals, weights, names_by_ticker)
ctx = build_ia_context(PROFIL, HORIZON, END_DATE, metrics, signals, weights, names_by_ticker)

# Génération du conseil
advice = generate_advice_openai(client, ctx)

print("\n=== CONSEIL IA ===\n")
print(advice)



=== CONSEIL IA ===

Profil et horizon : Équilibre, 12 mois.

- Idée rapide : Acheter Apple Inc. (AAPL).
- Pourquoi : La tendance est au-dessus de la moyenne sur 200 jours. Cela montre une bonne santé de l'action.
- Comment : Investir 50 % de votre budget maintenant et 50 % plus tard.
- À surveiller : Si le prix descend de 12 %, il faut réfléchir à vendre.

- Idée rapide : Acheter JPMorgan Chase & Co. (JPM).
- Pourquoi : L'action est aussi au-dessus de sa moyenne sur 200 jours, ce qui est positif.
- Comment : Même stratégie, 50 % maintenant et 50 % plus tard.
- À surveiller : Attention si le prix baisse de 12 %.

- Idée rapide : Acheter Microsoft Corporation (MSFT).
- Pourquoi : La tendance est bonne, au-dessus de la moyenne sur 200 jours.
- Comment : Investir 50 % tout de suite et 50 % plus tard.
- À surveiller : Réagir si le prix diminue de 12 %.

- Idée rapide : Acheter Exxon Mobil Corporation (XOM).
- Pourquoi : L'action est au-dessus de sa moyenne sur 200 jours, ce qui est encoura